# Suggested chunk shapes for ECCOFS

Opens the virtual Icechunk on source.coop and uses [`dynamic_chunks`](https://github.com/jbusecke/dynamic_chunks) (`iterative_ratio_increase_algo`) to suggest ~10 MB chunk shapes.
The **aspect ratio** is the desired relative number of chunks along (time, level, y, x); `-1` means "don't chunk this dimension".
Only shape and dtype matter, so nothing is read from the source apart from metadata.

In [ ]:
%pip install -q git+https://github.com/jbusecke/dynamic_chunks icechunk zarr xarray pandas matplotlib

In [ ]:
import math

import icechunk
import matplotlib.pyplot as plt
import pandas as pd
import xarray as xr
import zarr
from dynamic_chunks.algorithms import iterative_ratio_increase_algo

zarr.config.set({"async.concurrency": 32})

URL = "https://data.source.coop/ocean-icechunks/noaa-eccofs/eccofs-avg-best"
SOURCE = "s3://noaa-nos-eccofs-pds/"  # public bucket holding the NetCDF bytes

repo = icechunk.Repository.open(icechunk.http_storage(URL))
repo = repo.reopen(
    authorize_virtual_chunk_access=icechunk.containers_credentials({SOURCE: icechunk.s3_anonymous_credentials()})
)
ds = xr.open_zarr(repo.readonly_session("main").store, consolidated=False, chunks={})
ds[["temp", "u", "v"]]

## Helpers

In [ ]:
# Aspect ratio = desired relative number of chunks along (time, level, y, x); -1 = do not chunk that dim.
RATIOS = {
    "balanced": (1, 1, 1, 1),
    "map": (4, -1, 1, 1),   # full depth in one chunk, a few times
}


def suggest(da, ratio, target="10MB", tolerance=0.2):
    """Chunk shape (dict dim -> size) for DataArray `da`; `ratio` is ordered like da.dims."""
    one = da.to_dataset(name="v").drop_vars(list(da.coords), errors="ignore")
    return iterative_ratio_increase_algo(one, target, dict(zip(da.dims, ratio)), size_tolerance=tolerance)


def summarize(da, chunks):
    c = [chunks[d] for d in da.dims]
    per_dim = tuple(math.ceil(da.sizes[d] / k) for d, k in zip(da.dims, c))
    return {"chunks": tuple(c), "chunk MB": math.prod(c) * da.dtype.itemsize / 1e6,
            "chunks per dim": per_dim, "total chunks": math.prod(per_dim)}


def compare(da, ratios=RATIOS, **kw):
    """Table of the current chunking plus one suggestion per named ratio."""
    rows = {"current": summarize(da, dict(zip(da.dims, da.encoding["chunks"])))}
    for name, ratio in ratios.items():
        rows[name] = summarize(da, suggest(da, ratio, **kw))
    return pd.DataFrame(rows).T

## Variable to inspect

In [ ]:
var = "u"   # temp, u or v
da = ds[var]
print(f"{var}: dims={da.dims} shape={da.shape} dtype={da.dtype} total={da.nbytes/1e9:.0f} GB")

## Current chunking vs. suggestions

Add your own ratios to compare.

In [ ]:
ratios = {**RATIOS}   # e.g. ratios["time series"] = (-1, 1, 4, 4)
table = compare(da, ratios, target="10MB", tolerance=0.2)
table.style.format({"chunk MB": "{:.2f}"})

## What each layout looks like

Bars show chunk length as a fraction of the full dimension (full bar = dimension not chunked).

In [ ]:
fig, axes = plt.subplots(1, len(table), figsize=(3.2 * len(table), 3), sharey=True)
for ax, (name, row) in zip(axes, table.iterrows()):
    frac = [c / n for c, n in zip(row["chunks"], da.shape)]
    ax.barh(range(4), frac, color="tab:blue")
    ax.set_xscale("log"); ax.set_xlim(1e-3, 6)
    ax.set_yticks(range(4)); ax.set_yticklabels(da.dims); ax.invert_yaxis()
    for i, c in enumerate(row["chunks"]):
        ax.text(frac[i] * 1.15, i, str(c), va="center", fontsize=9)
    ax.set_title(f"{name}\n{row['chunk MB']:.1f} MB, {row['total chunks']:,} chunks", fontsize=10)
    ax.set_xlabel("chunk / dimension")
plt.tight_layout()

## Try your own ratio

In [ ]:
ratio = (-1, 1, 4, 4)   # (time, level, y, x)
try:
    chunks = suggest(da, ratio, target="10MB", tolerance=0.3)
    print(summarize(da, chunks))
except Exception as e:
    print("no match within tolerance:", e)